In [0]:
from utils.logger import get_logger
from etl_config.silver_config import SILVER_CONFIG, SILVER

logger = get_logger("silver_sanity_check")

In [0]:
for table_name, cfg in SILVER_CONFIG.items():
    # logger.info(f"{cfg.target_table}")
    # logger.info(f"{cfg.required_columns}")
    logger.info("==========================")
    logger.info(f"Table: {cfg.target_table}")
    logger.info("--------------------------")
    for column_name in cfg.required_columns:
        null_count = spark.sql(
            f"""
                select count(*) as null_count
                from {cfg.target_table}
                where {column_name} is null
            """
        ).collect()[0]["null_count"]

        logger.info(f"{column_name}: {null_count}")


In [0]:
# Orphan count

checks = [
    ["orders", "customer_id", "customers", "customer_id"],
    ["orders", "shipper_id", "shippers", "shipper_id"],
    ["order_details", "order_id", "orders", "order_id"],
    ["order_details", "product_id", "products", "product_id"],
    ["products", "category_id", "categories", "category_id"],
    ["customers", "division_id", "divisions", "division_id"],
    ["shipments", "order_id", "orders", "order_id"],
    ["shipments", "shipper_id", "shippers", "shipper_id"]
]

for child_table, child_column, parent_table, parent_column in checks:
    orphan_count = spark.sql(
        f"""
            select count(*) as orphan_count
            from {SILVER}.{child_table} c
            left anti join {SILVER}.{parent_table} p
            on c.{child_column} = p.{parent_column}
        """
    ).collect()[0]["orphan_count"]

    logger.info(f"{child_table}.{child_column} -> {parent_table}.{parent_column}: {orphan_count} orphans")
            

In [0]:
%sql
select distinct c.shipper_id 
from acme_catalog.silver.orders c
left anti join acme_catalog.silver.shippers p
    on c.shipper_id = p.shipper_id

In [0]:
%sql
select distinct c.shipper_id 
from acme_catalog.silver.shipments c
left anti join acme_catalog.silver.shippers p
    on c.shipper_id = p.shipper_id

In [0]:
%sql
-- Bad discount
select count(*) as cnt
from acme_catalog.silver.order_details
where discount < 0 or discount > 1

In [0]:
%sql
-- Bad quantity
select count(*) as cnt
from acme_catalog.silver.order_details
where quantity < 0;

In [0]:
%sql
-- Bad Price
select count(*) as cnt
from acme_catalog.silver.order_details
where unit_price <= 0;

In [0]:
%sql
-- Quarantine
select *
from acme_catalog.silver._quarantine;